In [ ]:
import os
import zipfile
import numpy as np
import pickle

In [ ]:
raw_data_dir = "/home/laura/Scriptie/ruwe_data"
bewerkte_data_dir = "/home/laura/Scriptie/bewerkte_data/segTHRawS"
os.makedirs(bewerkte_data_dir, exist_ok=True)

path_segTHRawS = "/home/laura/Scriptie/ruwe_data/Data_Hotspots.zip"

MAX_PATCHES = 100

all_X = []
all_y = []

try:
    with zipfile.ZipFile(path_segTHRawS, 'r') as outer_zip:
        with outer_zip.open('SegTHRawS_images_event.zip') as inner_images_file, \
             outer_zip.open('SegTHRawS_masks.zip') as inner_masks_file:
            with zipfile.ZipFile(inner_images_file) as z_img, zipfile.ZipFile(inner_masks_file) as z_mask:
                img_files = [f for f in z_img.namelist() if f.endswith('.pkl')]
                mask_files = [f for f in z_mask.namelist() if f.endswith('.pkl')]
                mask_dict = {os.path.basename(m).replace('_mask.pkl', ''): m for m in mask_files}

                done_patches = 0
                for img_file in img_files:
                    if done_patches >= MAX_PATCHES:
                        break
                    patch_name = os.path.basename(img_file).replace('.pkl', '')
                    if patch_name in mask_dict:
                        mask_path = mask_dict[patch_name]
                        try:
                            with z_img.open(img_file) as f:
                                img_data = pickle.load(f)
                            with z_mask.open(mask_path) as f:
                                mask_data = pickle.load(f)
                            if not isinstance(img_data, np.ndarray) or not isinstance(mask_data, np.ndarray):
                                print(f"Data in {img_file} or {mask_path} is not a numpy array.")
                                continue

                            X_flat = img_data.reshape(-1, 3)
                            y_flat = mask_data.reshape(-1)

                            all_X.append(X_flat)
                            all_y.append(y_flat)
                            done_patches += 1
                        except Exception as e:
                            print(f"Error processing {img_file} and {mask_path}: {e}")

    X_dataset = np.vstack(all_X)
    y_dataset = np.concatenate(all_y)

    print(f"\n --- TOTAL SegTHRawS DATASET ---")
    print(f" - X (Features) matrix    : {X_dataset.shape} (Features)")
    print(f" - y (Labels) list       : {y_dataset.shape} (Labels)")

    np.save(os.path.join(bewerkte_data_dir, 'X_train_flat.npy'), X_dataset)
    np.save(os.path.join(bewerkte_data_dir, 'y_train_flat.npy'), y_dataset)

except Exception as e:
    print(f"Error: {e}")

In [ ]:
raw_data_dir = '/home/laura/Scriptie/ruwe_data'
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/segTHRawS'
os.makedirs(bewerkte_data_dir, exist_ok=True) 

path_segTHRawS = os.path.join(raw_data_dir, 'Data_Hotspots.zip')
MAX_PATCHES = 100 

all_X = []
all_y = []

try:
    with zipfile.ZipFile(path_segTHRawS, 'r') as outer_zip:
        with outer_zip.open('SegTHRawS_images_event.zip') as inner_images_file, \
             outer_zip.open('SegTHRawS_masks.zip') as inner_masks_file:
                 
            with zipfile.ZipFile(inner_images_file) as z_img, zipfile.ZipFile(inner_masks_file) as z_mask:
                
                img_files = [f for f in z_img.namelist() if f.endswith('.pkl')]
                mask_files = [f for f in z_mask.namelist() if f.endswith('.pkl')]
                
                mask_dict = {os.path.basename(m).replace('_mask.pkl', ''): m for m in mask_files}
                
                processed = 0
                checked = 0
                
                for img_path in img_files:
                    if processed >= MAX_PATCHES:
                        break
                        
                    base_name = os.path.basename(img_path).replace('.pkl', '')
                    checked += 1
                    
                    if base_name not in mask_dict:
                        continue
                        
                    mask_path = mask_dict[base_name]
                    
                    try:
                        with z_img.open(img_path) as f:
                            img_data = pickle.load(f)
                        with z_mask.open(mask_path) as f:
                            mask_data = pickle.load(f)
                            
                        if not isinstance(img_data, np.ndarray) or not isinstance(mask_data, np.ndarray):
                            continue
                            
                        if not np.any(mask_data == 1):
                            continue
                        
                        X_flat = img_data.reshape(-1, 3)
                        y_flat = mask_data.reshape(-1)
                        
                        all_X.append(X_flat)
                        all_y.append(y_flat)
                        
                        processed += 1
                    except Exception as patch_error:
                        continue

    if len(all_X) > 0:
        X_dataset = np.vstack(all_X)
        y_dataset = np.concatenate(all_y)
        
        print(f"\n --- TOTAL SegTHRawS DATASET ---")
        print(f" - X (Features) matrix    : {X_dataset.shape}")
        print(f" - y (Labels) list        : {y_dataset.shape}")
        
        np.save(os.path.join(bewerkte_data_dir, 'X_train_flat_thraws.npy'), X_dataset)
        np.save(os.path.join(bewerkte_data_dir, 'y_train_flat_thraws.npy'), y_dataset)
        
    else:
        print("No fire found")

except Exception as e:
    print(f"Error {e}")